# File Handling

> Generated from [`04-File-Handling.py`](04-File-Handling.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Reading and writing files with `open()`, the `with` statement, file modes, binary vs text data, and memory-efficient handling of large files.

### Key points
- `open(path, mode)` returns a file object. Always use `with open(...) as f:` — it auto-closes the file even if an error occurs inside the block.
- Modes: `'r'` read (default), `'w'` write (overwrites!), `'a'` append, `'x'` exclusive create (errors if file exists), `'b'` binary, `'t'` text (default), `'+'` read+write.
- Reading: `f.read()` (whole file), `for line in f:` (memory-efficient, line by line), `f.readline()` (one line), `f.readlines()` (list of all lines).
- Writing: `f.write(str)` for one chunk, `f.writelines(list_of_strs)` for many at once (does NOT add newlines for you).
- `f.seek(offset)` moves the cursor; `f.tell()` reports its current position.
- Binary mode (`'rb'`/`'wb'`) works with `bytes` objects instead of `str` — use `.decode()`/`.encode()` to convert.
- `os.path.exists(path)` checks before opening; opening a missing file in `'r'` mode raises `FileNotFoundError`.
- `tempfile.NamedTemporaryFile()` creates a scratch file that's auto-deleted when the `with` block exits.
- `shutil.copy()` copies a file's content + permissions; `shutil.move()` relocates/renames.

### Gotchas (the part worth revising twice)
- **Always pass `encoding="utf-8"` explicitly.** Without it, Python uses the OS's default text encoding, which differs across machines — a file written on one machine can fail to read correctly on another.
- Buffering: default is a full OS-sized buffer, flushed when full or on close. `buffering=1` is line-buffered (text mode only, flushes on `\n`). `f.flush()` forces buffered data out immediately without closing — useful for logs you want visible right away.
- For **text** files, iterate line by line (`for line in f`) — memory-efficient because it never loads the whole file. For files without natural line breaks (binary, huge single-line data), read fixed-size chunks with `f.read(chunk_size)` in a loop instead — plain `f.read()` loads everything into memory at once.
- `'w'` mode silently overwrites/truncates existing content — easy to lose data by using the wrong mode.

### When it matters
Any time your program needs to persist data, read config/data files, process logs, or handle datasets too large to comfortably fit in memory.

### One-line recap example
```python
with open("data.txt", "r", encoding="utf-8") as f:
    for line in f:          # one line in memory at a time
        print(line.strip())
```

In [ ]:
import os

## 01. File Handling - Introduction

- Python reads/writes files via a file object returned by open().
- Always close files when done - `with` does this automatically.

## 02. File Modes (reference)

## 03. Writing to a File with `with` (auto-closes, even on error)

In [ ]:
with open("temp_demo.txt", "w") as f:
    f.write("Line 1\n")
    f.write("Line 2\n")
print("file written, closed automatically:", f.closed)

## 04. Reading an Entire File

In [ ]:
with open("temp_demo.txt", "r") as f:
    content = f.read()
print(repr(content))

## 05. Reading Line by Line

In [ ]:
with open("temp_demo.txt", "r") as f:
    for line in f:  # memory-efficient - reads one line at a time
        print("line:", line.strip())

## 06. readline() and readlines()

In [ ]:
with open("temp_demo.txt", "r") as f:
    first_line = f.readline()
    print("readline():", repr(first_line))

with open("temp_demo.txt", "r") as f:
    all_lines = f.readlines()
    print("readlines():", all_lines)

## 07. Appending to a File

In [ ]:
with open("temp_demo.txt", "a") as f:
    f.write("Line 3 (appended)\n")

with open("temp_demo.txt", "r") as f:
    print(f.read())

## 08. Writing Multiple Lines at Once

In [ ]:
lines = ["A\n", "B\n", "C\n"]
with open("temp_demo.txt", "w") as f:
    f.writelines(lines)
with open("temp_demo.txt", "r") as f:
    print(f.read())

## 09. seek() and tell() - Moving the File Cursor

In [ ]:
with open("temp_demo.txt", "r") as f:
    print("cursor position at start:", f.tell())
    f.read(2)
    print("cursor position after reading 2 chars:", f.tell())
    f.seek(0)  # move back to start
    print("after seek(0):", f.tell())

## 10. Binary File Handling

In [ ]:
with open("temp_demo.bin", "wb") as f:
    f.write(bytes([72, 101, 108, 108, 111]))  # "Hello" as bytes

with open("temp_demo.bin", "rb") as f:
    data = f.read()
print("binary data:", data)
print("decoded:", data.decode())

## 11. Checking Existence Before Opening

In [ ]:
if os.path.exists("temp_demo.txt"):
    print("temp_demo.txt exists")

try:
    open("does_not_exist.txt", "r")
except FileNotFoundError:
    print("Caught FileNotFoundError for missing file")

## 12. The encoding= Parameter - Avoiding Platform-Default Bugs

- Without encoding=, Python uses the OS's default text encoding, which
  differs across machines (e.g. UTF-8 on Linux/macOS, cp1252 on some
  Windows setups). A file written on one machine can fail to read
  correctly on another. ALWAYS pass encoding="utf-8" explicitly for text
  you care about being portable.

In [ ]:
with open("temp_demo.txt", "w", encoding="utf-8") as f:
    f.write("café, naïve, 日本語\n")  # non-ASCII characters
with open("temp_demo.txt", "r", encoding="utf-8") as f:
    print("read back correctly with explicit utf-8:", f.read().strip())

## 13. Buffering Modes - Line-Buffered vs Full, flush()

- buffering=1 means line-buffered (flushes after each newline, text mode
  only); buffering=0 is unbuffered (binary mode only); the default is a
  full buffer sized by the OS, flushed when the buffer fills or the file
  is closed.
- flush() forces buffered data to be written NOW, without closing the
  file - useful for logs you want visible immediately.

In [ ]:
with open("temp_demo.txt", "w", buffering=1, encoding="utf-8") as f:
    f.write("line-buffered write\n")
    print("data is likely on disk already (line-buffered flushes on \\n)")

with open("temp_demo.txt", "a", encoding="utf-8") as f:
    f.write("more data")
    f.flush()  # force it out without closing
    print("explicitly flushed before the `with` block even exits")

## 14. Memory-Efficient Large File Processing

- Reading line-by-line (`for line in f`) is memory-efficient for TEXT
  files with natural line breaks.
- For files without lines (binary, huge single-line data), read in fixed
  size CHUNKS with f.read(size) instead of f.read() (loads everything).

In [ ]:
with open("temp_demo.txt", "r", encoding="utf-8") as f:
    line_count = sum(1 for _ in f)  # one line in memory at a time
print("counted lines without loading the whole file:", line_count)

with open("temp_demo.bin", "wb") as f:
    f.write(bytes(range(256)) * 100)  # 25,600 bytes

chunk_size = 1024
bytes_seen = 0
with open("temp_demo.bin", "rb") as f:
    while chunk := f.read(chunk_size):  # read fixed-size chunks until empty
        bytes_seen += len(chunk)
print("counted bytes via chunked reads:", bytes_seen)

## 15. tempfile Module - Scratch Files

- tempfile creates files/directories in the OS temp location, with unique
  names, cleaned up automatically when used as a context manager.

In [ ]:
import tempfile

print("\n# 15. tempfile Module")
with tempfile.NamedTemporaryFile(mode="w+", suffix=".txt", delete=True) as tmp:
    tmp.write("scratch data")
    tmp.seek(0)
    print("wrote to and read from a temp file:", tmp.read())
    print("temp file path:", tmp.name)
print("temp file auto-deleted on exit from `with` block")

## 16. shutil Module - Copy/Move Files (brief)

- shutil.copy() copies a file's content + permissions; shutil.move()
  relocates (or renames) a file or directory.

In [ ]:
import shutil

print("\n# 16. shutil for Copy/Move")
shutil.copy("temp_demo.txt", "temp_demo_copy.txt")
print("copied file exists:", os.path.exists("temp_demo_copy.txt"))
shutil.move("temp_demo_copy.txt", "temp_demo_moved.txt")
print("after move - old name exists:", os.path.exists("temp_demo_copy.txt"))
print("after move - new name exists:", os.path.exists("temp_demo_moved.txt"))

## 17. CSV / Large-File Streaming (cross-reference)

- For structured data like CSV, don't hand-parse lines - the csv module
  handles quoting/escaping correctly, and can stream row-by-row just like
  the chunked reading above. Full depth (csv + json) lives in the
  Standard Library file - not duplicated here.

## 18. Cleanup

In [ ]:
os.remove("temp_demo.txt")
os.remove("temp_demo.bin")
os.remove("temp_demo_moved.txt")
print("\n# 18. Cleanup")
print("temp files removed")